In [1]:
import os
os.chdir('/home/smallyan/eval_agent')
print(f"Working directory: {os.getcwd()}")

Working directory: /home/smallyan/eval_agent


In [2]:
# Check for CUDA availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA device count: {torch.cuda.device_count()}")

CUDA available: True
CUDA device: NVIDIA A100 80GB PCIe
CUDA device count: 1


# Code Critic Evaluation: Leela Logit Lens Circuit Analysis

This notebook evaluates the implementation of the circuit analysis in the `leela-logit-lens_eval` repository.

## Evaluation Criteria

For each code block, we evaluate:
1. **Runnable (Y/N)**: Does the block execute without error?
2. **Correct-Implementation (Y/N)**: Is the logic correct relative to the stated purpose?
3. **Redundant (Y/N)**: Does the block duplicate another block's computation?
4. **Irrelevant (Y/N)**: Does the block contribute to the project goal?

---

## Setup and Environment

In [3]:
import os
import sys
import json
import warnings
warnings.filterwarnings('ignore')

# Set up paths
REPO_PATH = '/net/scratch2/smallyan/leela-logit-lens_eval'
os.chdir(REPO_PATH)
sys.path.insert(0, os.path.join(REPO_PATH, 'src'))

print(f"Working directory: {os.getcwd()}")
print(f"Python path includes: {REPO_PATH}/src")

# Check CUDA availability
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Working directory: /net/scratch2/smallyan/leela-logit-lens_eval
Python path includes: /net/scratch2/smallyan/leela-logit-lens_eval/src
Device: cuda
GPU: NVIDIA A100 80GB PCIe


In [4]:
# Create a data structure to track evaluation results
evaluation_results = []

def add_result(file_name, block_id, runnable, correct_impl, redundant, irrelevant, error_note=""):
    """Add an evaluation result for a code block."""
    evaluation_results.append({
        "file": file_name,
        "block_id": block_id,
        "runnable": runnable,
        "correct_implementation": correct_impl,
        "redundant": redundant,
        "irrelevant": irrelevant,
        "error_note": error_note if error_note else ""
    })
    status = "✓" if runnable == "Y" else "✗"
    print(f"{status} {file_name}:{block_id} - Runnable={runnable}, Correct={correct_impl}, Redundant={redundant}, Irrelevant={irrelevant}")
    if error_note:
        print(f"   Note: {error_note}")

print("Evaluation tracking initialized.")

Evaluation tracking initialized.


---

## 1. Core Library Evaluation: leela_logit_lens

Testing the core LeelaLogitLens class and its dependencies.

In [5]:
# Test 1: Import leela_interp (dependency)
try:
    from leela_interp import Lc0sight, LeelaBoard
    add_result("core/leela_interp", "import", "Y", "Y", "N", "N")
except Exception as e:
    add_result("core/leela_interp", "import", "N", "N", "N", "N", str(e))

✓ core/leela_interp:import - Runnable=Y, Correct=Y, Redundant=N, Irrelevant=N


In [6]:
# Test 2: Import LeelaLogitLens
try:
    from leela_logit_lens import LeelaLogitLens
    add_result("core/leela_logit_lens", "import", "Y", "Y", "N", "N")
except Exception as e:
    add_result("core/leela_logit_lens", "import", "N", "N", "N", "N", str(e))

✓ core/leela_logit_lens:import - Runnable=Y, Correct=Y, Redundant=N, Irrelevant=N


In [7]:
# Test 3: Load the model
try:
    model_path = "lc0-original.onnx"
    model = Lc0sight(model_path, device=device)
    add_result("core/model_loading", "Lc0sight_init", "Y", "Y", "N", "N")
except Exception as e:
    add_result("core/model_loading", "Lc0sight_init", "N", "N", "N", "N", str(e))

Using device: cuda
✗ core/model_loading:Lc0sight_init - Runnable=N, Correct=N, Redundant=N, Irrelevant=N
   Note: [Errno 2] No such file or directory: 'lc0-original.onnx'


In [8]:
# The model file is in the iteration_model directory. Let's update the last result and load from correct path.
# Remove the failed result and try again
evaluation_results.pop()

try:
    model_path = "iteration_model/lc0-original.onnx"
    model = Lc0sight(model_path, device=device)
    add_result("core/model_loading", "Lc0sight_init", "Y", "Y", "N", "N")
except Exception as e:
    add_result("core/model_loading", "Lc0sight_init", "N", "N", "N", "N", str(e))

Using device: cuda


✓ core/model_loading:Lc0sight_init - Runnable=Y, Correct=Y, Redundant=N, Irrelevant=N
